In [ ]:
import pyodbc
import pandas as pd

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=SotexHackathon;"
    "UID=sa;"
    "PWD=SotexSolutions123!;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

query = """
SELECT 
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH,
    IS_NULLABLE,
    CASE WHEN COLUMNPROPERTY(OBJECT_ID(TABLE_NAME), COLUMN_NAME, 'IsIdentity') = 1 THEN 'YES' ELSE 'NO' END as IS_IDENTITY
FROM INFORMATION_SCHEMA.COLUMNS
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

df = pd.read_sql(query, conn)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 30)

for table in df['TABLE_NAME'].unique():
    print("\n" + "═" * 90)
    print(f"  📁 TABELA: {table}")
    print("═" * 90)
    
    table_df = df[df['TABLE_NAME'] == table].copy()
    
    def format_type(row):
        if row['DATA_TYPE'] in ('nvarchar', 'varchar'):
            if pd.notna(row['CHARACTER_MAXIMUM_LENGTH']):
                if row['CHARACTER_MAXIMUM_LENGTH'] == -1:
                    return f"{row['DATA_TYPE']}(MAX)"
                else:
                    return f"{row['DATA_TYPE']}({int(row['CHARACTER_MAXIMUM_LENGTH'])})"
        return row['DATA_TYPE']
    
    table_df['DATA_TYPE'] = table_df.apply(format_type, axis=1)
    
    # Prikaz
    for _, row in table_df.iterrows():
        nullable = "NULL" if row['IS_NULLABLE'] == 'YES' else "NOT NULL"
        identity = "🔑 IDENTITY" if row['IS_IDENTITY'] == 'YES' else ""
        
        print(f"  • {row['COLUMN_NAME']:<25} {row['DATA_TYPE']:<20} {nullable:<10} {identity}")
    

conn.close()



══════════════════════════════════════════════════════════════════════════════════════════
  📁 TABELA: Channels
══════════════════════════════════════════════════════════════════════════════════════════
  • Id                        int                  NOT NULL   🔑 IDENTITY
  • Name                      nvarchar(100)        NULL       
  • Unit                      nvarchar(30)         NULL       

══════════════════════════════════════════════════════════════════════════════════════════
  📁 TABELA: DistributionSubstation
══════════════════════════════════════════════════════════════════════════════════════════
  • Id                        int                  NOT NULL   🔑 IDENTITY
  • Name                      nvarchar(100)        NULL       
  • MeterId                   int                  NULL       
  • Feeder11Id                int                  NULL       
  • Feeder33Id                int                  NULL       
  • NameplateRating           int                  NUL

C:\Users\Tanja\AppData\Local\Temp\ipykernel_23688\715110797.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
